In [1]:
# ============================================================================
# RETAIL GOLD ETL — MAIN PIPELINE
# ============================================================================
# Silver  →  Gold layer ETL for the Retail data warehouse.
#
# Silver layer has ONE schema (transaction_schema) — unlike the travel ETL
# which had master_schema + transaction_schema.  The master_schema parameter
# is accepted for interface compatibility but defaults to transaction_schema.
#
# Gold tables processed (in execution_order):
#   Dims  : dim_date, dim_customer, dim_store, dim_product,
#            dim_payment_method, dim_sales_channel
#   Facts : fact_sales, fact_sales_items, fact_payments,
#            fact_returns, fact_inventory
#   Agg   : agg_daily_sales_by_store, agg_monthly_sales_by_category,
#            agg_customer_summary, agg_product_performance,
#            agg_payment_method_summary, agg_sales_channel_performance,
#            agg_inventory_snapshot
#
# Usage:
#   result = run_gold_etl(
#       dl_host            = "localhost",
#       dl_port            = 5432,
#       dl_user            = "postgres",
#       dl_pass            = "password",
#       gold_db_name       = "gold_db",
#       silver_dbname      = "silver_db",
#       transaction_schema = "retail_silver",   # Silver schema name
#       load_type          = "DELTA",           # "DELTA" | "FULL"
#       run_frequency      = "DAILY",           # "DAILY" | "HOURLY" | "WEEKLY"
#       source_name        = "RetailSilver",
#       target_table_type  = None,              # None = all types
#       batch_size         = 100000,
#   )
# ============================================================================

import psycopg2
import pandas as pd
import logging
import re
import ast
from io import StringIO
from datetime import datetime, timedelta

pd.set_option('display.max_rows', 5)
logging.getLogger('sqlalchemy.engine').setLevel(logging.WARNING)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


# ============================================================================
# SECTION 1: DATABASE CONNECTION
# ============================================================================

def create_connection(host, port, user, password, db):
    conn = psycopg2.connect(
        host=host, port=port, user=user, password=password, dbname=db
    )
    logging.info(f"Database connection established: {db}")
    return conn


# ============================================================================
# SECTION 2: AUDIT LOG FUNCTIONS
# ============================================================================

def update_audit_master_tbl(
    connection,
    batch_run_id,
    etl_name,
    etl_start_time=None,
    etl_end_time=None,
    status=None,
    table_processed=0,
    table_succeeded=0,
    table_failed=0,
    error_msg=None,
    process_id=None,
    etl_schedule_time=None,
):
    """
    Inserts (status='in_progress') or updates the master audit record in
    gold_etl_run_master_audit_tbl_test.

    Returns:
        int: master_audit_id of the inserted / updated record.
    """
    try:
        cursor = connection.cursor()

        if status and status.lower() == "in_progress":
            insert_query = """
                INSERT INTO gold_etl_master.gold_etl_run_master_audit_tbl_test (
                    etl_batch_id,
                    schedule_name,
                    schedule_timestamp,
                    status,
                    start_time,
                    end_time,
                    tables_processed,
                    tables_successful,
                    tables_failed,
                    error_message
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                RETURNING master_audit_id
            """
            cursor.execute(insert_query, (
                int(batch_run_id), etl_name, etl_schedule_time, status,
                etl_start_time, etl_end_time,
                int(table_processed), int(table_succeeded), int(table_failed),
                error_msg,
            ))
            process_id = cursor.fetchone()[0]
            logging.info(f"New ETL process started — Process ID: {process_id}")

        else:
            if process_id is None:
                raise Exception("Cannot update audit without a Process ID")
            update_query = """
                UPDATE gold_etl_master.gold_etl_run_master_audit_tbl_test
                SET status            = %s,
                    end_time          = %s,
                    tables_processed  = %s,
                    tables_successful = %s,
                    tables_failed     = %s,
                    error_message     = %s
                WHERE master_audit_id = %s
            """
            cursor.execute(update_query, (
                status, etl_end_time,
                int(table_processed), int(table_succeeded), int(table_failed),
                error_msg, int(process_id),
            ))
            logging.info(f"ETL process completed — Process ID: {process_id}")

        connection.commit()
        return process_id

    except Exception as e:
        connection.rollback()
        msg = f"Failed to update audit master table: {e}"
        logging.error(msg)
        raise Exception(msg)


def upsert_etl_audit_record(
    connection,
    process_id,
    batch_run_id,
    source_name,
    destination_table,
    operation,
    time_from,
    status,
    input_row=0,
    time_to=None,
    output_row=0,
    error_code=None,
    result=None,
    etl_end_time=None,
    etl_start_time=None,
    target_schema_name=None,
):
    """
    Inserts (status='in-progress') or updates a table-level audit record in
    gold_etl_run_table_audit_log.
    """
    try:
        input_row  = int(input_row)  if input_row  is not None else 0
        output_row = int(output_row) if output_row is not None else 0

        with connection.cursor() as cur:
            if status.lower() in ("in-progress", "in_progress"):
                cur.execute("""
                    INSERT INTO gold_etl_master.gold_etl_run_table_audit_log (
                        process_id, batch_run_id, data_source, db_code,
                        destination_table_name, operation,
                        time_from, time_to,
                        etl_start_time, etl_end_time,
                        input_row, output_row,
                        operation_result, errorcode, result
                    ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                """, (
                    int(process_id), int(batch_run_id),
                    source_name, target_schema_name,
                    destination_table, operation,
                    time_from, time_to,
                    etl_start_time, time_to,
                    input_row, output_row,
                    status, error_code, result,
                ))
                logging.info(f"[AUDIT] In-progress logged: {destination_table}")
            else:
                cur.execute("""
                    UPDATE gold_etl_master.gold_etl_run_table_audit_log
                    SET time_to          = %s,
                        etl_end_time     = %s,
                        input_row        = %s,
                        output_row       = %s,
                        operation_result = %s,
                        errorcode        = %s,
                        result           = %s
                    WHERE process_id             = %s
                      AND batch_run_id           = %s
                      AND destination_table_name = %s
                """, (
                    time_to, etl_end_time,
                    input_row, output_row,
                    status, error_code, result,
                    int(process_id), int(batch_run_id), destination_table,
                ))
                logging.info(f"[AUDIT] Completion logged: {destination_table}")

        connection.commit()

    except Exception as e:
        msg = f"Audit log failed for '{destination_table}': {e}"
        logging.error(msg)
        connection.rollback()
        raise Exception(msg)


# ============================================================================
# SECTION 3: GLOBAL BATCH AUDIT FUNCTIONS
# ============================================================================

def create_global_batch_record(conn, load_type, run_frequency, run_start_datetime,
                               source_name=None):
    """
    Creates a RUNNING row in gold_etl_global_batch_audit.
    source_name is ignored (no such column in the DDL).

    Returns:
        int: batch_run_id
    """
    try:
        with conn.cursor() as cur:
            cur.execute("""
                INSERT INTO gold_etl_master.gold_etl_global_batch_audit (
                    run_start_datetime, run_end_datetime,
                    run_status, load_type, run_frequency,
                    is_batch_processed, is_reprocessed
                ) VALUES (%s,%s,%s,%s,%s,%s,%s)
                RETURNING batch_run_id
            """, (
                run_start_datetime,
                run_start_datetime,   # placeholder — overwritten on completion
                'RUNNING', load_type, run_frequency,
                False, False,
            ))
            batch_run_id = cur.fetchone()[0]
        conn.commit()
        logging.info(
            f"Global batch created — batch_run_id: {batch_run_id}, "
            f"load_type: {load_type}, frequency: {run_frequency}"
        )
        return batch_run_id
    except Exception as e:
        conn.rollback()
        msg = f"Failed to create global batch record: {e}"
        logging.error(msg)
        raise Exception(msg)


def update_global_batch_record(conn, batch_run_id, run_status, run_end_datetime):
    """
    Updates run_status + run_end_datetime in gold_etl_global_batch_audit.
    Sets is_batch_processed=True when COMPLETED.
    """
    try:
        is_done = run_status == "COMPLETED"
        with conn.cursor() as cur:
            cur.execute("""
                UPDATE gold_etl_master.gold_etl_global_batch_audit
                SET run_status         = %s,
                    run_end_datetime   = %s,
                    is_batch_processed = %s,
                    updated_at         = NOW()
                WHERE batch_run_id     = %s
            """, (run_status, run_end_datetime, is_done, int(batch_run_id)))
        conn.commit()
        logging.info(f"Global batch {batch_run_id} → {run_status}")
    except Exception as e:
        conn.rollback()
        msg = f"Failed to update global batch {batch_run_id}: {e}"
        logging.error(msg)
        raise Exception(msg)


# ============================================================================
# SECTION 4: CONFIGURATION READING
# ============================================================================

def get_gold_layer_etl_config(
    connection,
    source=None,
    target_table_type=None,
    target_table_names=None,
    run_type=None,
    run_frequency=None,
):
    """
    Reads active rows from gold_layer_etl_config_tbl filtered by the
    supplied parameters.
    """
    def _safe_eval(v):
        if isinstance(v, (list, tuple)):
            return v
        try:
            return ast.literal_eval(v)
        except Exception:
            return v

    query  = """
        SELECT *
        FROM gold_etl_master.gold_layer_etl_config_tbl
        WHERE is_config_active = TRUE
    """
    params = []

    if source:
        source = _safe_eval(source)
        if isinstance(source, (list, tuple)) and source:
            query += " AND source IN (" + ",".join(["%s"] * len(source)) + ")"
            params.extend(source)
        else:
            query += " AND source = %s"
            params.append(source)

    if target_table_type:
        query += " AND target_table_type = %s"
        params.append(target_table_type)

    if target_table_names:
        target_table_names = _safe_eval(target_table_names)
        if isinstance(target_table_names, (list, tuple)) and target_table_names:
            query += " AND target_table_name IN (" + ",".join(["%s"] * len(target_table_names)) + ")"
            params.extend(target_table_names)
        else:
            query += " AND target_table_name = %s"
            params.append(target_table_names)

    if run_type and run_type.upper() == "DELTA":
        if run_frequency:
            query += " AND run_frequency = %s"
            params.append(run_frequency)
        else:
            logging.warning("run_frequency not provided for DELTA run — including all frequencies")

    query += " ORDER BY execution_order"
    df = pd.read_sql(query, connection, params=params)
    logging.info(f"ETL config rows retrieved: {len(df)}")
    return df


# ============================================================================
# SECTION 5: UTILITY FUNCTIONS
# ============================================================================

def get_last_processed_datetime(gold_connection, table_name):
    """
    Returns the most recent successful time_to for a given destination table
    (DELTA runs only).
    """
    query = """
        SELECT getal.time_to
        FROM gold_etl_master.gold_etl_run_table_audit_log  getal
        JOIN gold_etl_master.gold_etl_global_batch_audit   gegba
            ON gegba.batch_run_id = getal.batch_run_id
        WHERE getal.destination_table_name = %s
          AND gegba.load_type              = 'DELTA'
          AND getal.operation_result       = 'Success'
        ORDER BY getal.time_to DESC
        LIMIT 1
    """
    try:
        with gold_connection.cursor() as cur:
            cur.execute(query, (table_name,))
            result = cur.fetchone()
            return result[0] if result and result[0] else None
    except Exception as e:
        logging.error(f"Could not fetch last processed datetime for {table_name}: {e}")
        return None


def replace_schema_placeholders(sql_query, transaction_schema):
    """
    Retail Silver has a SINGLE schema (transaction_schema).
    All {silver_schema}."<table>" placeholders are replaced uniformly.

    The function also handles the dual-placeholder syntax from the travel ETL
    ({master_schema} and {transaction_schema}) for forward compatibility, but
    in practice the retail config uses only {silver_schema}.
    """
    # Pattern 1: "{master_schema}"."<table>"  → transaction_schema (retail has no separate master)
    sql_query = re.sub(
        r'"\{master_schema\}"\."([^"]+)"',
        lambda m: f'"{transaction_schema}"."{m.group(1)}"',
        sql_query,
    )
    # Pattern 2: "{transaction_schema}"."<table>"
    sql_query = re.sub(
        r'"\{transaction_schema\}"\."([^"]+)"',
        lambda m: f'"{transaction_schema}"."{m.group(1)}"',
        sql_query,
    )
    # Pattern 3: legacy "{silver_schema}"."<table>"
    sql_query = re.sub(
        r'"\{silver_schema\}"\."([^"]+)"',
        lambda m: f'"{transaction_schema}"."{m.group(1)}"',
        sql_query,
    )
    # Fallback: bare {silver_schema}
    sql_query = sql_query.replace("{silver_schema}", transaction_schema)
    return sql_query


def replace_time_bounds_in_query(sql_query, run_type, lower_bound=None, upper_bound=None):
    """
    Replaces {lower_bound_time}, {upper_bound_time} (and legacy variants) in
    the SQL template with real datetime values.

    For FULL runs the lower bound defaults to '2000-01-01'.
    """
    import re as _re

    def _fix_double_quotes(sql):
        return _re.sub(
            r"''(\d{4}-\d{2}-\d{2}(?:[T ][\d:.]+)?)''"
            , r"'\1'", sql,
        )

    # ── Upper bound ──────────────────────────────────────────────────────────
    if upper_bound:
        ub_str = (
            upper_bound.strftime('%Y-%m-%d %H:%M:%S')
            if isinstance(upper_bound, datetime) else str(upper_bound)
        )
        for ph in ("{upper_bound_time}", "{upper_bound_date}"):
            if ph in sql_query:
                sql_query = sql_query.replace(ph, f"'{ub_str}'")
                logging.info(f"Upper bound → {ub_str}")

    # ── Lower bound ──────────────────────────────────────────────────────────
    lower_placeholders = (
        "{lower_bound_time}", "{lower_bound_date}", "{last_processed_date}"
    )
    if run_type.upper() == "DELTA":
        if lower_bound:
            lb_str = (
                lower_bound.strftime('%Y-%m-%d %H:%M:%S')
                if isinstance(lower_bound, datetime) else str(lower_bound)
            )
            for ph in lower_placeholders:
                if ph in sql_query:
                    sql_query = sql_query.replace(ph, f"'{lb_str}'")
                    logging.info(f"Lower bound → {lb_str}")
        else:
            for ph in lower_placeholders:
                if ph in sql_query:
                    logging.warning(f"Lower bound placeholder '{ph}' has no value for DELTA run")
    elif run_type.upper() == "FULL":
        for ph in lower_placeholders:
            if ph in sql_query:
                sql_query = sql_query.replace(ph, "'2000-01-01'")
                logging.info("Lower bound defaulted to 2000-01-01 for FULL run")

    return _fix_double_quotes(sql_query)


def add_order_by_clause(sql_query, load_key):
    """Appends ORDER BY if missing; strips any baked-in LIMIT."""
    if not load_key:
        return sql_query
    sql_query = re.sub(r';\s*$', '', sql_query.rstrip())
    sql_query = re.sub(r'\bLIMIT\s+\d+\s*$', '', sql_query, flags=re.IGNORECASE).rstrip()
    if not re.search(r'\border\s+by\b', sql_query, re.IGNORECASE):
        cols = [c.strip() for c in load_key.split(',')]
        sql_query += " ORDER BY " + ", ".join(cols)
    return sql_query


def clean_dataframe_for_db(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepares a DataFrame for PostgreSQL COPY:
        - Converts whole-number floats → Int64 (nullable integer).
        - Converts all-null float columns → object so they write as empty string.
        - Replaces NaN/None with None (psycopg2 COPY NULL '').
    """
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]):
            non_null = df[col].notna()
            if non_null.sum() == 0:
                df[col] = df[col].astype(object)
            elif df.loc[non_null, col].apply(lambda x: float(x).is_integer()).all():
                df[col] = df[col].astype('Int64')
    df = df.astype(object).where(pd.notna(df), None)
    return df


def generate_incremental_where_clause(incremental_keys, lower_bound_time):
    """Builds a WHERE clause for APPEND-mode delete of rows newer than lower_bound."""
    if not incremental_keys or not lower_bound_time:
        return ""
    if isinstance(lower_bound_time, datetime):
        lower_bound_time = lower_bound_time.strftime('%Y-%m-%d %H:%M:%S')
    conditions   = [f'"{k}" > \'{lower_bound_time}\'' for k in incremental_keys]
    return "WHERE " + " OR ".join(conditions)


# ============================================================================
# SECTION 6: DATA LOADING
# ============================================================================

def load_to_gold(df_chunk, target_schema_name, target_table_name, gold_conn,
                 ingestion_strategy="TRUNCATE_INSERT", conflict_columns=None):
    """
    Writes a DataFrame chunk to a Gold PostgreSQL table.

    Strategies
    ----------
    TRUNCATE_INSERT / APPEND  →  PostgreSQL COPY (fast bulk load)
    MERGE                     →  INSERT … ON CONFLICT DO UPDATE (upsert)
    """
    if df_chunk.empty:
        return

    import csv as _csv
    df_chunk        = clean_dataframe_for_db(df_chunk)
    table_full_name = f'"{target_schema_name}"."{target_table_name}"'

    if ingestion_strategy in ("TRUNCATE_INSERT", "APPEND"):
        buffer = StringIO()
        df_chunk.to_csv(
            buffer, index=False, header=False,
            sep='\t', na_rep='', quoting=_csv.QUOTE_MINIMAL,
        )
        buffer.seek(0)
        columns  = ', '.join([f'"{c}"' for c in df_chunk.columns])
        copy_sql = (
            f"COPY {table_full_name} ({columns}) FROM STDIN "
            f"WITH (FORMAT CSV, DELIMITER E'\\t', NULL '', HEADER FALSE)"
        )
        try:
            with gold_conn.cursor() as cur:
                cur.copy_expert(copy_sql, buffer)
            gold_conn.commit()
        except Exception as e:
            gold_conn.rollback()
            logging.exception(f"COPY failed for {table_full_name}: {e}")
            raise

    elif ingestion_strategy == "MERGE":
        if not conflict_columns:
            raise ValueError("conflict_columns required for MERGE strategy")
        columns = list(df_chunk.columns)
        conflict_cols = (
            [c.strip() for c in conflict_columns.split(',')]
            if isinstance(conflict_columns, str) else list(conflict_columns)
        )
        insert_cols  = ', '.join([f'"{c}"' for c in columns])
        placeholders = ', '.join(['%s'] * len(columns))
        conflict_cl  = ', '.join([f'"{c}"' for c in conflict_cols])
        update_set   = ', '.join([
            f'"{c}" = EXCLUDED."{c}"'
            for c in columns if c not in conflict_cols
        ])
        sql = (
            f"INSERT INTO {table_full_name} ({insert_cols}) "
            f"VALUES ({placeholders}) "
            f"ON CONFLICT ({conflict_cl}) DO UPDATE SET {update_set}"
        )
        try:
            with gold_conn.cursor() as cur:
                cur.executemany(sql, df_chunk.values.tolist())
            gold_conn.commit()
        except Exception as e:
            gold_conn.rollback()
            logging.exception(f"MERGE failed for {table_full_name}: {e}")
            raise
    else:
        raise ValueError(f"Unknown ingestion_strategy: {ingestion_strategy}")


# ============================================================================
# SECTION 7: TABLE PROCESSING
# ============================================================================

def process_table_delta(
    silver_conn, gold_conn, row,
    lower_bound_time, upper_bound_time,
    transaction_schema,
    batch_size=50000,
):
    """
    DELTA (incremental) load for a single table.

    Steps:
        1. Replace {silver_schema} placeholders with transaction_schema.
        2. Replace time-bound placeholders.
        3. APPEND: delete rows >= lower_bound then re-insert.
           TRUNCATE_INSERT: truncate then bulk-insert.
        4. Paginated extraction + batch load into Gold.

    Returns:
        tuple: (total_fetched, total_processed, final_status, error_message)
    """
    sql_query          = row['data_load_sql_template']
    sql_query          = replace_schema_placeholders(sql_query, transaction_schema)
    sql_query          = replace_time_bounds_in_query(
        sql_query, 'DELTA', lower_bound_time, upper_bound_time
    )
    logging.info(f"[SQL] {sql_query[:300]}...")

    target_table_name  = row['target_table_name']
    target_schema_name = row['target_schema_name']
    ingestion_strategy = row['ingestion_strategy']
    load_key           = row['load_key']
    inc_key_str        = str(row['incremental_extract_key'])
    incremental_keys   = [k.strip() for k in inc_key_str.split(',') if k.strip()]

    total_fetched   = 0
    total_processed = 0
    final_status    = "failed"
    error_message   = None
    offset          = 0
    batch_count     = 0

    try:
        # ── Pre-load table management ─────────────────────────────────────────
        if ingestion_strategy.upper() == "APPEND" and incremental_keys:
            where_clause = generate_incremental_where_clause(
                incremental_keys, lower_bound_time
            )
            if where_clause:
                delete_sql = (
                    f'DELETE FROM "{target_schema_name}"."{target_table_name}" {where_clause}'
                )
                with gold_conn.cursor() as cur:
                    cur.execute(delete_sql)
                    deleted = cur.rowcount
                    gold_conn.commit()
                logging.info(
                    f"APPEND pre-delete: {deleted} rows removed from {target_table_name}"
                )

        if ingestion_strategy.upper() == "TRUNCATE_INSERT":
            with gold_conn.cursor() as cur:
                cur.execute(
                    f'TRUNCATE TABLE "{target_schema_name}"."{target_table_name}" '
                    f'RESTART IDENTITY CASCADE'
                )
            gold_conn.commit()
            logging.info(f"Truncated: {target_table_name}")

        # ── Paginated extraction + load ───────────────────────────────────────
        ordered_sql = add_order_by_clause(sql_query, load_key)

        while True:
            paginated_sql = f"{ordered_sql} LIMIT {batch_size} OFFSET {offset}"
            try:
                df_chunk = pd.read_sql_query(paginated_sql, silver_conn)
            except Exception as e:
                error_message = f"Extraction query failed: {e}"
                logging.error(error_message)
                raise

            if df_chunk.empty:
                if batch_count == 0:
                    logging.warning(f"No data found for {target_table_name}")
                    error_message = "No data found in source"
                final_status = "success"
                break

            batch_count   += 1
            total_fetched += len(df_chunk)

            try:
                load_to_gold(
                    df_chunk, target_schema_name, target_table_name, gold_conn,
                    ingestion_strategy, conflict_columns=load_key,
                )
                total_processed += len(df_chunk)
            except Exception as e:
                error_message = f"Load failed on batch {batch_count}: {e}"
                logging.error(error_message)
                raise

            offset += len(df_chunk)
            if len(df_chunk) < batch_size:
                final_status = "success"
                break

    except Exception as e:
        if error_message is None:
            error_message = f"Unexpected error: {e}"
        final_status = "failed"

    logging.info(
        f"{target_table_name} → {batch_count} batch(es), "
        f"{total_processed} rows, status: {final_status}"
    )
    return total_fetched, total_processed, final_status, error_message


def process_table_full(
    silver_conn, gold_conn, row,
    upper_bound_time, transaction_schema,
    batch_size=50000,
):
    """
    FULL load (truncate + reload) for a single table.

    Returns:
        tuple: (total_fetched, total_processed, final_status, error_message)
    """
    sql_query          = row['data_load_sql_template']
    sql_query          = replace_schema_placeholders(sql_query, transaction_schema)
    sql_query          = replace_time_bounds_in_query(
        sql_query, 'FULL', None, upper_bound_time
    )
    target_table_name  = row['target_table_name']
    target_schema_name = row['target_schema_name']
    load_key           = row['load_key']

    total_fetched   = 0
    total_processed = 0
    final_status    = "failed"
    error_message   = None
    offset          = 0
    batch_count     = 0

    try:
        with gold_conn.cursor() as cur:
            cur.execute(
                f'TRUNCATE TABLE "{target_schema_name}"."{target_table_name}" '
                f'RESTART IDENTITY CASCADE'
            )
        gold_conn.commit()
        logging.info(f"Truncated: {target_table_name}")

        ordered_sql = add_order_by_clause(sql_query, load_key)

        while True:
            paginated_sql = f"{ordered_sql} LIMIT {batch_size} OFFSET {offset}"
            try:
                df_chunk = pd.read_sql_query(paginated_sql, silver_conn)
            except Exception as e:
                error_message = f"Extraction query failed: {e}"
                logging.error(error_message)
                raise

            if df_chunk.empty:
                if batch_count == 0:
                    logging.warning(f"No data found for {target_table_name}")
                    error_message = "No data found in source"
                final_status = "success"
                break

            batch_count   += 1
            total_fetched += len(df_chunk)

            try:
                load_to_gold(
                    df_chunk, target_schema_name, target_table_name, gold_conn,
                    ingestion_strategy="TRUNCATE_INSERT",
                )
                total_processed += len(df_chunk)
            except Exception as e:
                error_message = f"Load failed on batch {batch_count}: {e}"
                logging.error(error_message)
                raise

            offset += len(df_chunk)
            if len(df_chunk) < batch_size:
                final_status = "success"
                break

    except Exception as e:
        if error_message is None:
            error_message = f"Unexpected error: {e}"
        final_status = "failed"

    logging.info(
        f"{target_table_name} → {batch_count} batch(es), "
        f"{total_processed} rows, status: {final_status}"
    )
    return total_fetched, total_processed, final_status, error_message


# ============================================================================
# SECTION 8: SUMMARY LOGGING
# ============================================================================

def log_etl_summary(
    start_time, end_time,
    total_tables, successful_tables, failed_tables, failed_table_names,
    total_records_fetched, total_records_processed,
):
    duration     = end_time - start_time
    success_rate = (successful_tables / total_tables * 100) if total_tables > 0 else 0
    logging.info("=" * 70)
    logging.info("RETAIL GOLD ETL — RUN SUMMARY")
    logging.info("=" * 70)
    logging.info(f"Start Time         : {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    logging.info(f"End Time           : {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
    logging.info(f"Duration           : {duration}")
    logging.info(f"Total Tables       : {total_tables}")
    logging.info(f"Successful Tables  : {successful_tables}")
    logging.info(f"Failed Tables      : {failed_tables}")
    logging.info(f"Success Rate       : {success_rate:.1f}%")
    logging.info(f"Records Fetched    : {total_records_fetched}")
    logging.info(f"Records Processed  : {total_records_processed}")
    if failed_table_names:
        logging.info("FAILED TABLES:")
        for i, t in enumerate(failed_table_names, 1):
            logging.info(f"  {i}. {t}")
    status_msg = "COMPLETED SUCCESSFULLY" if failed_tables == 0 else f"COMPLETED WITH {failed_tables} FAILURE(S)"
    logging.info(f"STATUS             : {status_msg}")
    logging.info("=" * 70)


# ============================================================================
# SECTION 9: MAIN ORCHESTRATION
# ============================================================================

def run_gold_etl(
    dl_host,
    dl_port,
    dl_user,
    dl_pass,
    gold_db_name,
    silver_dbname,
    transaction_schema,                      # Retail Silver schema (e.g. "retail_silver")
    master_schema=None,                      # Not used for retail; accepted for compat
    load_type="DELTA",
    run_frequency="DAILY",
    etl_name="RETAIL_GOLD_ETL_RUN",
    source_name="RetailSilver",
    target_table_type=None,
    target_table_names=None,
    batch_size=100000,
):
    """
    Main entry point for the Retail Gold ETL pipeline.

    Reads active config from gold_etl_master.gold_layer_etl_config_tbl,
    then processes each table (dims → facts → aggs) in execution_order.

    Args
    ----
    dl_host            (str)  : DB host for both Gold and Silver.
    dl_port            (int)  : DB port.
    dl_user            (str)  : DB user.
    dl_pass            (str)  : DB password.
    gold_db_name       (str)  : Gold DB name.
    silver_dbname      (str)  : Silver DB name.
    transaction_schema (str)  : Silver schema holding retail tables.
    master_schema      (str)  : Accepted but unused (retail has one schema).
    load_type          (str)  : "DELTA" or "FULL".
    run_frequency      (str)  : "DAILY" | "HOURLY" | "WEEKLY".
    etl_name           (str)  : Label written to audit records.
    source_name        (str)  : Source filter for config table lookup.
    target_table_type  (str)  : Optional filter: "dim" | "fact" | "agg".
    target_table_names (list) : Optional list of specific target table names.
    batch_size         (int)  : Rows per paginated extraction chunk.

    Returns
    -------
    dict with keys: status, batch_run_id, etl_run_id, total_tables,
                    successful_tables, failed_tables, failed_table_names,
                    total_records_fetched, total_records_processed.
    """
    # master_schema defaults to transaction_schema for retail (single schema)
    if master_schema is None:
        master_schema = transaction_schema

    etl_run_id    = None
    batch_run_id  = None
    etl_start     = datetime.now()
    upper_bound   = etl_start       # snapshot time — constant for this run

    total_tables     = 0
    successful_tables = 0
    failed_tables     = 0
    failed_names      = []
    total_fetched     = 0
    total_processed   = 0

    logging.info("=" * 70)
    logging.info("STARTING RETAIL GOLD LAYER ETL")
    logging.info("=" * 70)
    logging.info(f"ETL Name           : {etl_name}")
    logging.info(f"Load Type          : {load_type}")
    logging.info(f"Run Frequency      : {run_frequency}")
    logging.info(f"Transaction Schema : {transaction_schema}")
    logging.info(f"Upper Bound Time   : {upper_bound}")

    # ── Connections ────────────────────────────────────────────────────────────
    gold_conn   = create_connection(dl_host, dl_port, dl_user, dl_pass, gold_db_name)
    silver_conn = create_connection(dl_host, dl_port, dl_user, dl_pass, silver_dbname)

    try:
        # ── STEP 1: Read config ────────────────────────────────────────────────
        logging.info("STEP 1: Reading ETL config...")
        config_df = get_gold_layer_etl_config(
            connection=gold_conn,
            source=source_name,
            target_table_type=target_table_type,
            target_table_names=target_table_names,
            run_type=load_type,
            run_frequency=run_frequency,
        )

        if config_df.empty:
            msg = "No active ETL config found — nothing to process."
            logging.warning(msg)
            return {"status": "Aborted", "message": msg}

        config_df = config_df.sort_values("execution_order").reset_index(drop=True)
        logging.info(f"Tables to process  : {len(config_df)}")
        logging.info(f"Execution order    : {config_df['target_table_name'].tolist()}")

        # ── STEP 2: Validate load_type ─────────────────────────────────────────
        if load_type not in ("DELTA", "FULL"):
            raise Exception(
                f"Invalid load_type '{load_type}'. Expected 'DELTA' or 'FULL'."
            )

        # ── STEP 3: Global batch record ────────────────────────────────────────
        logging.info("STEP 2: Creating global batch record...")
        batch_run_id = create_global_batch_record(
            conn=gold_conn,
            load_type=load_type,
            run_frequency=run_frequency,
            run_start_datetime=etl_start,
            source_name=str(source_name),
        )

        # ── STEP 4: Master audit record ────────────────────────────────────────
        logging.info("STEP 3: Creating master audit record...")
        etl_run_id = update_audit_master_tbl(
            connection=gold_conn,
            batch_run_id=batch_run_id,
            etl_name=etl_name,
            etl_start_time=etl_start,
            status="in_progress",
            etl_schedule_time=etl_start,
        )
        logging.info(f"ETL Run ID: {etl_run_id}")

        # ── STEP 5: Process each table ─────────────────────────────────────────
        logging.info("=" * 70)
        logging.info(f"STEP 4: Processing tables [{load_type}]")
        logging.info("=" * 70)

        for _, row in config_df.iterrows():
            total_tables    += 1
            table_start      = datetime.now()
            table_name       = row["target_table_name"]
            source_current   = row["source"]
            target_schema    = row["target_schema_name"]
            operation        = row["ingestion_strategy"]

            logging.info(
                f"\n[{total_tables}/{len(config_df)}] {table_name} "
                f"| strategy: {operation}"
            )

            # Table audit — in-progress
            upsert_etl_audit_record(
                gold_conn, etl_run_id, batch_run_id,
                source_current, table_name, operation,
                time_from=table_start, status="in-progress",
                etl_start_time=etl_start, target_schema_name=target_schema,
            )

            tbl_fetched    = 0
            tbl_processed  = 0
            tbl_status     = "success"
            tbl_error      = None

            try:
                if load_type == "DELTA":
                    lower_bound = get_last_processed_datetime(gold_conn, table_name)
                    if lower_bound:
                        lower_bound -= timedelta(minutes=5)   # 5-min overlap for safety
                        logging.info(f"Lower bound: {lower_bound}")
                    else:
                        lower_bound = datetime(2000, 1, 1)
                        logging.info(
                            f"No prior run found for {table_name} — "
                            f"defaulting to {lower_bound}"
                        )

                    tbl_fetched, tbl_processed, tbl_status, tbl_error = process_table_delta(
                        silver_conn=silver_conn,
                        gold_conn=gold_conn,
                        row=row,
                        lower_bound_time=lower_bound,
                        upper_bound_time=upper_bound,
                        transaction_schema=transaction_schema,
                        batch_size=batch_size,
                    )

                elif load_type == "FULL":
                    tbl_fetched, tbl_processed, tbl_status, tbl_error = process_table_full(
                        silver_conn=silver_conn,
                        gold_conn=gold_conn,
                        row=row,
                        upper_bound_time=upper_bound,
                        transaction_schema=transaction_schema,
                        batch_size=batch_size,
                    )

                total_fetched    += tbl_fetched
                total_processed  += tbl_processed

            except Exception as exc:
                tbl_status = "failed"
                tbl_error  = f"Exception on '{table_name}': {exc}"
                logging.error(tbl_error)
                failed_tables += 1
                failed_names.append(table_name)

            else:
                if tbl_status == "success":
                    successful_tables += 1
                    logging.info(f"✓ {table_name} — OK")
                else:
                    failed_tables += 1
                    failed_names.append(table_name)
                    logging.error(f"✗ {table_name} — FAILED: {tbl_error}")

            # Table audit — final
            upsert_etl_audit_record(
                gold_conn, etl_run_id, batch_run_id,
                source_current, table_name, operation,
                time_from=table_start,
                status="Success" if tbl_status == "success" else "Failed",
                input_row=tbl_fetched,
                time_to=upper_bound,
                output_row=tbl_processed,
                error_code=tbl_error,
                result="Success" if tbl_status == "success" else "Failed",
                etl_end_time=datetime.now(),
                etl_start_time=etl_start,
                target_schema_name=target_schema,
            )

        # ── STEP 6: Finalise ───────────────────────────────────────────────────
        etl_end        = datetime.now()
        overall_status = "Success"   if failed_tables == 0 else "Failed"
        run_status     = "COMPLETED" if failed_tables == 0 else "FAILED"

        log_etl_summary(
            etl_start, etl_end, total_tables, successful_tables,
            failed_tables, failed_names, total_fetched, total_processed,
        )

        update_audit_master_tbl(
            connection=gold_conn, batch_run_id=batch_run_id, etl_name=etl_name,
            etl_end_time=etl_end, status=overall_status,
            table_processed=total_tables, table_succeeded=successful_tables,
            table_failed=failed_tables,
            error_msg=(
                None if not failed_names
                else f"Failed tables: {', '.join(failed_names)}"
            ),
            process_id=etl_run_id,
        )

        update_global_batch_record(
            conn=gold_conn, batch_run_id=batch_run_id,
            run_status=run_status, run_end_datetime=etl_end,
        )

        logging.info(f"STEP 5: Pipeline complete — {run_status}")

        return {
            "status"                  : overall_status,
            "batch_run_id"            : batch_run_id,
            "etl_run_id"              : etl_run_id,
            "total_tables"            : total_tables,
            "successful_tables"       : successful_tables,
            "failed_tables"           : failed_tables,
            "failed_table_names"      : failed_names,
            "total_records_fetched"   : total_fetched,
            "total_records_processed" : total_processed,
        }

    except Exception as main_exc:
        etl_end = datetime.now()
        logging.error(f"Pipeline failed: {main_exc}")

        if etl_run_id and batch_run_id:
            try:
                update_audit_master_tbl(
                    connection=gold_conn, batch_run_id=batch_run_id,
                    etl_name=etl_name, etl_end_time=etl_end, status="Failed",
                    table_processed=total_tables, table_succeeded=successful_tables,
                    table_failed=failed_tables, error_msg=str(main_exc),
                    process_id=etl_run_id,
                )
            except Exception as ae:
                logging.error(f"Could not update master audit: {ae}")

        if batch_run_id:
            try:
                update_global_batch_record(
                    conn=gold_conn, batch_run_id=batch_run_id,
                    run_status="FAILED", run_end_datetime=etl_end,
                )
            except Exception as be:
                logging.error(f"Could not update global batch: {be}")

        if total_tables > 0:
            log_etl_summary(
                etl_start, etl_end, total_tables, successful_tables,
                failed_tables, failed_names, total_fetched, total_processed,
            )

        raise Exception(f"Retail Gold ETL failed: {main_exc}")

    finally:
        gold_conn.close()
        silver_conn.close()
        logging.info("Connections closed.")
        logging.info("=" * 70)
        logging.info("RETAIL GOLD ETL — DONE")
        logging.info("=" * 70)


In [ ]:
# ============================================================================
# EXAMPLE INVOCATION
# ============================================================================
if __name__ == "__main__":python -m venv venv 
    result = run_gold_etl(
        dl_host            = "localhost",
        dl_port            = 5432,
        dl_user            = "postgres",
        dl_pass            = "password",
        gold_db_name       = "retail_gold_db",
        silver_dbname      = "retail_silver_db",
        transaction_schema = "retail_silver",   # your Silver schema name
        load_type          = "DELTA",           # or "FULL" for initial load
        run_frequency      = "DAILY",
        source_name        = "RetailSilver",
        target_table_type  = None,              # None = all dims + facts + aggs
        batch_size         = 100_000,
    )

    print("\nETL Result:")
    for k, v in result.items():
        print(f"  {k:<30}: {v}")